# Manual Gegenpressing Annotation

Notebook ini membuat ground truth perilaku gegenpressing secara terpisah dari outcome quick regain. Queue menargetkan 40 transisi strict-valid per pertandingan, seimbang antara quick regain dan non-regain. Sampel dipisahkan minimal dua detik agar window keputusan tidak memiliki titik awal yang bertumpuk. Jika kuota tidak tersedia, jumlah pertandingan diturunkan secara simetris. Outcome disembunyikan dari panel keputusan untuk mengurangi bias anotator.

Metodologi turnover, window struktural, dan kriteria strict-valid dari notebook 05 tidak diubah.

## Definisi operasional yang digunakan

Unit keputusan adalah dua detik pertama setelah kehilangan penguasaan (`T0` sampai `T0+2s`). Literatur mendefinisikan counterpressing sebagai tekanan segera setelah kehilangan bola terhadap pembawa bola atau lawan di sekitar bola. Regain dalam lima detik adalah ukuran **keberhasilan**, bukan definisi keberadaan gegenpressing.

Beri `is_gegenpressing = 1` jika terlihat reaksi tekanan segera dan disengaja, yaitu sedikitnya satu pemain losing team menutup pembawa bola/bola, atau beberapa pemain menutup ruang dan opsi umpan di sekitar bola. Tekanan harus tampak berkelanjutan, bukan hanya kebetulan berada dekat bola.

Beri `is_gegenpressing = 0` jika respons dominan adalah mundur/reorganisasi, tidak ada pemain yang segera menekan, atau kedekatan pemain tidak disertai gerak menutup lawan/ruang.

Beri `-1` (ragu) untuk bola udara/clearance, pembawa bola tidak dapat diidentifikasi, transisi terlalu singkat, atau bukti visual tidak cukup. Sampel ragu tidak dipakai sebagai kelas training sampai ditinjau ulang.

Panel menampilkan cue, bukan label otomatis:
- tekanan langsung: jarak ke bola <= 4,572 m (lima yard);
- approaching pressure: jarak <= 10 m dan jarak berkurang >= 1 m/s;
- durasi cue minimum: 0,4 detik;
- jumlah pemain dalam radius 10 m/15 m dan compactness lokal;
- lingkaran hijau menandai pemain yang memenuhi cue, sedangkan lingkaran gelap menandai proxy pembawa bola (lawan terdekat ke bola).

Ambang selain lima yard adalah parameter audit awal dan harus dievaluasi melalui agreement/analisis sensitivitas, bukan diperlakukan sebagai kebenaran taktis. Dasar metodologi: [Bauer & Anzer (2021)](https://link.springer.com/article/10.1007/s10618-021-00763-7).

In [1]:
from functools import lru_cache
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.gegenpressing_annotation import (
    AnnotationConfig,
    annotation_progress,
    compute_annotation_context,
    initialize_annotation_manifest,
    plot_annotation_context,
    record_annotation,
)
from src.idsse_episode_dataset import load_match

DATA_DIR = PROJECT_ROOT / 'data' / 'sportec_idsse'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EPISODE_MANIFEST_PATH = PROCESSED_DIR / 'episode_manifest.csv'
ANNOTATION_PATH = PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv'
CONFIG = AnnotationConfig()
ANNOTATOR = 'rizal'


In [2]:
annotations = initialize_annotation_manifest(
    EPISODE_MANIFEST_PATH,
    ANNOTATION_PATH,
    samples_per_match=40,
    random_state=42,
    min_spacing_seconds=2.0,
)

queue_check = (
    annotations.groupby(['match_id', 'quick_regain_success'])
    .size()
    .unstack(fill_value=0)
)
display(queue_check)
display(annotation_progress(annotations).to_frame('count'))
print('Annotation file:', ANNOTATION_PATH)

quick_regain_success,0,1
match_id,,
J03WMX,20,20
J03WN1,12,12
J03WOH,20,20
J03WOY,20,20
J03WPY,20,20
J03WQQ,20,20
J03WR9,20,20


,count
total_queue,264
completed,0
remaining,264
gegenpressing,0
non_gegenpressing,0
uncertain,0


Annotation file: c:\VAEP-Track\data\processed\gegenpressing_annotation_manifest.csv


## Panel anotasi

Tentukan label dari pola gerak dan cue yang terlihat. Jangan membuka kolom `quick_regain_success` untuk memutuskan label. Confidence: `1` rendah, `2` sedang, `3` tinggi. Setiap klik langsung menyimpan CSV sehingga progres tidak hilang.

In [3]:
@lru_cache(maxsize=1)
def load_cached_match(match_id):
    return load_match(DATA_DIR, match_id)

candidate_selector = widgets.IntSlider(
    value=1, min=1, max=len(annotations), step=1,
    description='Candidate', continuous_update=False,
    layout=widgets.Layout(width='70%'),
)
confidence = widgets.ToggleButtons(
    options=[('Rendah', 1), ('Sedang', 2), ('Tinggi', 3)],
    value=2, description='Confidence',
)
notes = widgets.Textarea(
    description='Catatan',
    placeholder='Alasan keputusan atau penyebab ragu',
    layout=widgets.Layout(width='85%', height='70px'),
)
positive_button = widgets.Button(description='Gegenpressing', button_style='success')
negative_button = widgets.Button(description='Bukan', button_style='danger')
uncertain_button = widgets.Button(description='Ragu', button_style='warning')
previous_button = widgets.Button(description='Sebelumnya')
next_button = widgets.Button(description='Berikutnya')
view_output = widgets.Output()
status_output = widgets.Output()

def current_index():
    return int(candidate_selector.value - 1)

def show_current(*_):
    idx = current_index()
    row = annotations.iloc[idx]
    tracking, events = load_cached_match(row['match_id'])
    context = compute_annotation_context(row, tracking, events, CONFIG)
    with view_output:
        clear_output(wait=True)
        fig = plot_annotation_context(context)
        display(fig)
        plt.close(fig)
        display(context['summary'].to_frame('value'))
        existing = row['is_gegenpressing']
        print(f"Tersimpan: {existing if pd.notna(existing) else 'belum'}")
    notes.value = '' if pd.isna(row['annotation_notes']) else str(row['annotation_notes'])
    if pd.notna(row['annotation_confidence']):
        confidence.value = int(row['annotation_confidence'])
    with status_output:
        clear_output(wait=True)
        display(annotation_progress(annotations).to_frame('count'))

def save_label(label):
    idx = current_index()
    record_annotation(
        annotations, idx, label, confidence.value, notes.value,
        ANNOTATOR, ANNOTATION_PATH,
    )
    if candidate_selector.value < candidate_selector.max:
        candidate_selector.value += 1
    else:
        show_current()

positive_button.on_click(lambda _: save_label(1))
negative_button.on_click(lambda _: save_label(0))
uncertain_button.on_click(lambda _: save_label(-1))
previous_button.on_click(lambda _: setattr(
    candidate_selector, 'value', max(candidate_selector.min, candidate_selector.value - 1)
))
next_button.on_click(lambda _: setattr(
    candidate_selector, 'value', min(candidate_selector.max, candidate_selector.value + 1)
))
candidate_selector.observe(show_current, names='value')

display(candidate_selector)
display(confidence, notes)
display(widgets.HBox([previous_button, negative_button, uncertain_button, positive_button, next_button]))
display(status_output, view_output)
show_current()

IntSlider(value=1, continuous_update=False, description='Candidate', layout=Layout(width='70%'), max=264, min=…

ToggleButtons(description='Confidence', index=1, options=(('Rendah', 1), ('Sedang', 2), ('Tinggi', 3)), value=…

Textarea(value='', description='Catatan', layout=Layout(height='70px', width='85%'), placeholder='Alasan keput…

Output()

Output()

## Audit setelah anotasi

Jalankan cell berikut secara berkala. Hubungan dengan quick regain baru dibuka pada tahap audit, setelah keputusan perilaku dibuat.

In [4]:
saved = pd.read_csv(ANNOTATION_PATH)
display(annotation_progress(saved).to_frame('count'))

completed = saved[saved['is_gegenpressing'].notna()].copy()
if len(completed):
    completed['is_gegenpressing'] = completed['is_gegenpressing'].astype(int)
    display(pd.crosstab(
        completed['is_gegenpressing'],
        completed['quick_regain_success'],
        margins=True,
    ))
    display(pd.crosstab(
        completed['match_id'],
        completed['is_gegenpressing'],
        margins=True,
    ))
else:
    print('Belum ada label manual yang tersimpan.')

,count
total_queue,264
completed,0
remaining,264
gegenpressing,0
non_gegenpressing,0
uncertain,0


Belum ada label manual yang tersimpan.


## Quality gate sebelum training

Sebelum label dipakai untuk training:
1. Tinjau ulang semua label `-1` dan confidence rendah.
2. Minta anotator kedua menilai minimal 50-60 sampel tanpa melihat label pertama.
3. Laporkan agreement (persentase agreement dan Cohen's kappa).
4. Bekukan versi manifest dan jangan mengubah label berdasarkan outcome quick regain.
5. Gunakan split berbasis pertandingan, bukan random split episode.